In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Preprocessing

### Packages

In [ ]:
%%time
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

### Load the Data

In [ ]:
%%time
# Load datasets
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
sample_submission_df = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

# Show first few rows
print(train_df.head())
#print(s_submission_df.head())

📓 Starter Notebook: TF-IDF + Logistic Regression + try quick make submission

In [ ]:
%%time
# 📦 Imports
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 📥 Load Data
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
submission_df = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")

# 🧹 Text Preprocessing Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"<.*?>", "", text)    # Remove HTML tags
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)  # Remove punctuation
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 🧼 Clean the text
train_df['text_clean'] = train_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

# 📊 Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text_clean'], train_df['target'], test_size=0.2, random_state=42, stratify=train_df['target'])

# 🔠 TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test_df['text_clean'])

# 🤖 Model Training
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# 📈 Validation Prediction & F1
val_preds = model.predict(X_val_tfidf)
print("Validation F1 Score:", f1_score(y_val, val_preds))

# 📤 Predict on Test Set
test_preds = model.predict(X_test_tfidf)

# 📁 Create Submission File
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': test_preds
})
submission.to_csv("submission.csv", index=False)

Then re-vectorize on text_final.

### 🟨 2. Use Class Weights in Logistic Regression
To address the imbalance slightly:

In [ ]:
%%time
model = LogisticRegression(max_iter=1000, class_weight='balanced')

##  Feature Engineering

### 🔮 Recommended Next Step: Add Pretrained GloVe Embeddings
Why?

Pretrained GloVe vectors help your LSTM understand semantic meaning of words

Boosts F1 without changing the model architecture

GloVe vectors are compact and work well on Kaggle (no GPU issues)


🔽 Step 1: Download GloVe (100D)

Use Kaggle’s built-in datasets. Run this cell at the top:

In [ ]:
%%time
!cp /kaggle/input/glove6b100dtxt/glove.6B.100d.txt ./glove.6B.100d.txt

In [ ]:
%%time
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
Embedding(input_dim=20000, output_dim=64)

## Clean Datasets and try to train

In [ ]:
%%time
# 📦 Imports
import pandas as pd
import numpy as np
import re
import string
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# 📥 Load data
train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

# 🧹 Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train['keyword'] = train['keyword'].fillna('none')
test['keyword'] = test['keyword'].fillna('none')

train['text_clean'] = train['keyword'] + ' ' + train['text'].apply(clean_text)
test['text_clean'] = test['keyword'] + ' ' + test['text'].apply(clean_text)

# 🔠 Tokenization
tokenizer = Tokenizer(num_words=20000, oov_token='<OOV>')
tokenizer.fit_on_texts(train['text_clean'])

X = tokenizer.texts_to_sequences(train['text_clean'])
X = pad_sequences(X, maxlen=100)
X_test = tokenizer.texts_to_sequences(test['text_clean'])
X_test = pad_sequences(X_test, maxlen=100)
y = train['target'].values

# 🧪 Train-validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 🧠 Build model
model = Sequential([
    Embedding(input_dim=20000, output_dim=64, input_length=100),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ⏱️ Train
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=128, callbacks=[early_stop])

# 📈 Validation F1 Score
val_preds = (model.predict(X_val) > 0.5).astype(int)
print("Validation F1 Score:", f1_score(y_val, val_preds))


✅ Here's what to do before 

Step 2 (fix for tokenizer not defined):

Insert this Step 1.5 code before loading GloVe:

In [ ]:
%%time
from sklearn.model_selection import train_test_split

# Load the original train.csv
import pandas as pd
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

# Optional: clean the text column
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text_clean"] = train_df["text"].apply(clean_text)
test_df["text_clean"] = test_df["text"].apply(clean_text)

# Split into training and validation sets
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df["target"])


In [ ]:
%%time
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

## Visualize

### 📊 1. Training History Plot

Visualize how loss and accuracy changed during training:

In [ ]:
%%time
import matplotlib.pyplot as plt

def plot_training_history(history):
    plt.figure(figsize=(14, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_training_history(history)


📌 2. Confusion Matrix

Understand how the model performs on each class:

In [ ]:
%%time
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_val, val_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Disaster', 'Disaster'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix on Validation Set')
plt.show()

🧠 (Optional) 3. Word Cloud by Class

See which words appear most in disaster vs non-disaster tweets:

In [ ]:
%%time
from wordcloud import WordCloud

def plot_wordcloud_by_class(df, label):
    text = " ".join(df[df['target'] == label]['text_clean'])
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Target = {label}')
    plt.show()

plot_wordcloud_by_class(train_df, 1)  # Disaster tweets
plot_wordcloud_by_class(train_df, 0)  # Non-disaster tweets

## Validation

🧠 Step 2: Load GloVe into an Embedding Matrix

Embedding layer (with GloVe weights)

Bidirectional LSTM or Conv1D

Dense + Dropout

Output layer with sigmoid


#### ✅ Step 3: Tokenization + Padding + GloVe Embeddings

In [ ]:
%%time
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Parameters
MAX_NUM_WORDS = 20000
MAX_SEQUENCE_LENGTH = 100
EMBEDDING_DIM = 100

# Tokenize
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text_clean"])

X_train = tokenizer.texts_to_sequences(train_df["text_clean"])
X_val = tokenizer.texts_to_sequences(val_df["text_clean"])
X_test = tokenizer.texts_to_sequences(test_df["text_clean"])

X_train = pad_sequences(X_train, maxlen=MAX_SEQUENCE_LENGTH)
X_val = pad_sequences(X_val, maxlen=MAX_SEQUENCE_LENGTH)
X_test = pad_sequences(X_test, maxlen=MAX_SEQUENCE_LENGTH)

y_train = train_df["target"].values
y_val = val_df["target"].values


#### ✅ Step 3B: Load GloVe into Embedding Matrix

In [ ]:
%%time
# Load GloVe vectors
embedding_index = {}
with open("/kaggle/input/glove6b100dtxt/glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

# Create embedding matrix
word_index = tokenizer.word_index
num_words = min(MAX_NUM_WORDS, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= MAX_NUM_WORDS:
        continue
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

#### ✅ Step 4: Define the Keras Model (with GloVe Embeddings)
This architecture uses:

Pretrained GloVe embeddings (non-trainable)

Bidirectional LSTM for sequence understanding

Dropout for regularization

Dense layers for classification

In [ ]:
%%time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=False  # freeze GloVe
    ),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

### ✅ Step 5: Train the Model

Now run this block to train:

In [ ]:
%%time
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

### ✅ Step 6: Evaluate with F1 Score

Once training finishes, evaluate it like this:

In [ ]:
%%time
from sklearn.metrics import f1_score

val_preds = (model.predict(X_val) > 0.5).astype(int)
f1 = f1_score(y_val, val_preds)
print("Validation F1 Score:", f1)

Let’s now finish the pipeline by generating predictions on the test set and creating a Kaggle submission file.

## Submission

🧠 Step 1: Prepare the Test Set

In [ ]:
%%time
# Tokenize and pad the test text
max_len = 10
X_test = tokenizer.texts_to_sequences(test_df['text'])
X_test = pad_sequences(X_test, maxlen=max_len)

🤖 Step 2: Predict on Test Set

In [ ]:
%%time
# Predict probabilities
test_preds = model.predict(X_test, batch_size=512, verbose=1)

# Convert probabilities to binary predictions
test_preds_binary = (test_preds > 0.5).astype(int).flatten()

📄 Step 3: Create Submission File

In [ ]:
%%time
submission_df = sample_submission_df.copy()
submission_df['target'] = test_preds_binary
submission_df.to_csv('submission.csv', index=False)

# Display preview
submission_df.head()